# AI Face Images — Group K-Fold Cross Validation (Identity/Video-aware)

งานนี้มี CSV ไฟล์เดียว **ไม่ต้อง merge** แต่ต้องแบ่ง fold แบบระวัง **identity / video leakage**:
รูปหลายรูปอาจมาจากคนเดียวกันหรือวิดีโอเดียวกัน (เช่น หลายเฟรมจาก deepfake video, หรือรูปหลายรูปของคนเดียวใน imdb_wiki)
ถ้าไม่ group ให้ถูก โมเดลอาจ "จำหน้า" แทนที่จะเรียนรู้ลักษณะ fake/real จริงๆ

**การตั้งค่าตามที่ยืนยัน:**
- Group K-Fold โดยดึง group key จาก `Image Path` (identity/video id)
- Stratify ด้วยคอลัมน์ `Target` (real/fake) เท่านั้น
- 10-fold


In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_colwidth", None)


## 1. Config

แก้ `CSV_PATH` ให้ตรงกับตำแหน่งไฟล์จริงบนเซิร์ฟเวอร์
(`IMAGE_ROOT` ใช้เฉพาะถ้าต้องการเช็คว่าไฟล์รูปมีอยู่จริง — ใส่ `None` เพื่อข้ามขั้นตอนนี้ได้)

In [ ]:
CSV_PATH = Path("/path/to/your_ai_face_dataset.csv")   # TODO: แก้ path ให้ตรงจริง
IMAGE_ROOT = None   # เช่น Path("/media/hdd3/AI_Face_imagesV2_root") หรือ None ถ้าไม่เช็คไฟล์

N_SPLITS = 10
RANDOM_STATE = 42


## 2. โหลดข้อมูล

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]  # กันเผื่อมีช่องว่างใน header

print(f"Total rows: {len(df)}")
print(f"Columns   : {list(df.columns)}")
print(f"\nTarget distribution:")
print(df["Target"].value_counts())
df.head()


## 3. สร้าง Group Key จาก Image Path

แต่ละหมวดของข้อมูลมีรูปแบบ path ต่างกัน ทำให้วิธีดึง "identity / video id" ต้องต่างกันไปด้วย:

| หมวด | ตัวอย่าง path | Group key ที่ดึง |
|---|---|---|
| `DMs/DCFACe/{id}/{n}.jpg` | โฟลเดอร์เลขคือ identity | `DCFACe_{id}` |
| `deepfakes/celebdf/.../..._Celeb-synthesis_idA_idB_clip_frame.png` | คู่ id + clip คือวิดีโอเดียวกัน (frame ต่างกัน) | `celebdf_Celeb-synthesis_idA_idB_clip` |
| `deepfakes/celebdf/.../..._YouTube-real_clip_frame.png` | clip คือวิดีโอเดียวกัน | `celebdf_YouTube-real_clip` |
| `deepfakes/ff++/.../..._youtube_vid_frame.png` | vid คือวิดีโอเดียวกัน | `ffpp_youtube_vid` |
| `deepfakes/ff++/.../..._{Method}_srcA_srcB_frame(N).png` (Method เช่น Deepfakes, FaceSwap, Face2Face, NeuralTextures — รองรับชื่อที่มีตัวเลขปนและไฟล์ซ้ำแบบ `(2)`) | คู่ id คือวิดีโอเดียวกัน | `ffpp_{Method}_srcA_srcB` |
| `deepfakes/dfdc/(real|fake)/..._vid_frame.png` | vid คือวิดีโอเดียวกัน | `dfdc_vid` |
| `deepfakes/dfd/(real|fake)/..._actors__scenario__hash_frame.png` | หลายเฟรมจากวิดีโอเดียวกัน (DeepFakeDetection) | `dfd_actors__scenario__hash` |
| `deepfakes/celebdf/real/..._Celeb-real_id{n}_{clip}_frame.png` | คู่ id+clip คือวิดีโอเดียวกัน | `celebdf_Celeb-real_id{n}_{clip}` |
| `DMs/CommercialTools/{tool}/..._{uuid}_....png` (มี UUID ฝังในชื่อไฟล์ เช่น midjourney) | UUID เดียวกัน = generation เดียวกัน | `commercial_{tool}_{uuid}` |
| `DMs/CommercialTools/{tool}/{id}_face...` (ไม่มี UUID) | id ต่อ tool | `commercial_{tool}_{id}` |
| `.../nmXXXXXXX_rmXXXXXXXXXX_{date-like}_YYYY_faceN...` (ที่ไหนก็ได้ ไม่จำกัด folder) | nm id คือตัวคนจาก IMDB | `imdbwiki_nmXXXXXXX` |
| `.../XXXXXXXX_{date-like}_YYYY_faceN...` (ที่ไหนก็ได้ ไม่จำกัด folder, date-like ยอมรับรูปแบบวันที่เพี้ยน/สั้นได้) | เลข id คือตัวคนจาก Wiki-crop — **จับข้าม folder โดยตั้งใจ** เพราะพบว่า `DMs/StableDiffusion1.5`, `DMs/StableDiffusion_Inpainting` ใช้ id ชุดเดียวกับ `Real/imdb_wiki/wiki` (แปลว่าเป็นรูป generate/inpaint จากคนจริงคนเดียวกัน ถ้าไม่ group ข้าม folder จะเกิด identity leakage ข้าม Target class ได้) | `imdbwiki_wiki_XXXXXXXX` |
| อื่นๆ (GANs/ProGAN, StyleGAN, StyleGAN2/SFHQ, stylegan3, taming_transformer_VQGAN, latent_diffusion, Real/FFHQ, AttGAN, Palette, STARGAN, STGAN, MSG_STYLE_GAN, MMD_GAN ฯลฯ) | รูป synthetic ที่ generate อิสระต่อกัน (sequential id/seed เฉยๆ ไม่ใช่ identity) ไม่มี identity ซ้ำ — ตรวจสอบตัวอย่างจริงแล้วว่าไม่มี id ซ้ำ | ใช้ path ทั้งบรรทัดเป็น group เดี่ยว (unique) |

**สำคัญ:** รูปแบบด้านบนอิงจากตัวอย่างข้อมูลจริงที่เจอแล้วเท่านั้น ข้อมูลเต็ม (1.3M แถว) อาจมี pattern อื่นที่ยังไม่เจอ
ให้ตรวจสอบผลลัพธ์ในเซลล์ verification ด้านล่าง (โดยเฉพาะ breakdown ของ `unmatched_unique` ตาม top-level folder) แล้วเพิ่ม regex ใหม่ได้ถ้าเจอ folder ที่ควรมี identity ซ้ำแต่ยังไม่ถูกจับ

In [ ]:
PATTERNS = [
    # (label, regex, group_template)
    ("DCFACe",
     r"DMs/DCFACe/(?P<id>\d+)/",
     "DCFACe_{id}"),

    ("celebdf_synthesis",
     r"deepfakes/celebdf/[^/]+/\d+_(?P<name>Celeb-synthesis_id\d+_id\d+_\d+)_\d+(?:\(\d+\))?\.(?:png|jpg|jpeg)$",
     "celebdf_{name}"),

    ("celebdf_real_youtube",
     r"deepfakes/celebdf/[^/]+/\d+_(?P<name>YouTube-real_\d+)_\d+(?:\(\d+\))?\.(?:png|jpg|jpeg)$",
     "celebdf_{name}"),

    ("celebdf_real_celeb",
     r"deepfakes/celebdf/[^/]+/\d+_(?P<name>Celeb-real_id\d+_\d+)_\d+(?:\(\d+\))?\.(?:png|jpg|jpeg)$",
     "celebdf_{name}"),

    ("ffpp_youtube",
     r"deepfakes/ff\+\+/[^/]+/\d+_(?P<name>youtube_\d+)_\d+(?:\(\d+\))?\.(?:png|jpg|jpeg)$",
     "ffpp_{name}"),

    ("ffpp_manipulated",
     # method name อาจมีตัวเลขปน (เช่น Face2Face) และไฟล์อาจมี "(2)" ต่อท้ายกรณีซ้ำ/แก้ไข
     r"deepfakes/ff\+\+/[^/]+/\d+_(?P<name>[A-Za-z0-9]+_\d+_\d+)_\d+(?:\(\d+\))?\.(?:png|jpg|jpeg)$",
     "ffpp_{name}"),

    ("dfdc",
     r"deepfakes/dfdc/(?:real|fake)/\d+_(?P<id>\d+)_\d+\.",
     "dfdc_{id}"),

    ("dfd",
     r"deepfakes/dfd/(?:real|fake)/\d+_(?P<name>.+)_\d+\.(?:png|jpg|jpeg)$",
     "dfd_{name}"),

    # ไม่จำกัด folder โดยตั้งใจ: id เดียวกันอาจถูกใช้ซ้ำใน tool อื่น
    # (เช่น StableDiffusion1.5 / StableDiffusion_Inpainting ใช้ wiki id เดิม)
    # ต้อง group ให้ตรงกันข้าม folder เพื่อกัน identity leakage ข้าม Target class
    # ส่วนวันที่ยอมให้หลวมกว่าปกติ (digits/./- ความยาว 6-14 ตัวอักษร) เพราะเจอวันที่เพี้ยน
    # ในข้อมูลจริง เช่น "1989-08-019", "11-03-00", "1994-11-18."
    ("imdb_generic",
     r"(?P<id>nm\d+)_rm\d+_[\d\.\-]{6,14}_\d{4}_face",
     "imdbwiki_{id}"),

    ("wiki_generic",
     r"(?<!nm)(?P<id>\d+)_[\d\.\-]{6,14}_\d{4}_face",
     "imdbwiki_wiki_{id}"),

    ("commercial_uuid",
     # midjourney ฯลฯ ฝัง UUID ของ generation ไว้ในชื่อไฟล์ -> UUID เดียวกันคือรูปจาก generation เดียวกัน
     r"DMs/CommercialTools/(?P<tool>[^/]+)/.*?(?P<uuid>[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})",
     "commercial_{tool}_{uuid}"),

    ("commercial_numeric_id",
     r"DMs/CommercialTools/(?P<tool>[^/]+)/(?P<id>\d+)_",
     "commercial_{tool}_{id}"),
]

COMPILED_PATTERNS = [(label, re.compile(pattern), tmpl) for label, pattern, tmpl in PATTERNS]


def extract_group(path: str, row_idx: int):
    for label, regex, tmpl in COMPILED_PATTERNS:
        m = regex.search(path)
        if m:
            return tmpl.format(**m.groupdict()), label
    # ไม่เข้า pattern ไหนเลย -> ถือว่าเป็นภาพ synthetic อิสระ ไม่มี identity ซ้ำ
    return f"unique_{row_idx}", "unmatched_unique"


groups_and_labels = [
    extract_group(path, idx) for idx, path in enumerate(df["Image Path"])
]
df["group_key"] = [g for g, _ in groups_and_labels]
df["group_pattern"] = [lbl for _, lbl in groups_and_labels]

print("จำนวน record แยกตาม pattern ที่จับได้:")
print(df["group_pattern"].value_counts())


## 4. ตรวจสอบผลการดึง Group Key

- ดูตัวอย่างว่า group ไหนมีมากกว่า 1 รูป (ควรเป็นวิดีโอ/คนที่มีหลายเฟรมจริง)
- ดูตัวอย่างแถวที่ `unmatched_unique` เพื่อยืนยันว่าเป็นภาพ synthetic ที่ควรเป็น unique จริง ไม่ใช่ pattern ที่ยังไม่ได้เขียน regex รองรับ

In [ ]:
group_sizes = df.groupby("group_key").size().sort_values(ascending=False)

print("Top 15 group ที่มีจำนวนรูปมากที่สุด:")
print(group_sizes.head(15))

print(f"\nจำนวน group ทั้งหมด : {df['group_key'].nunique()}")
print(f"จำนวนรูปทั้งหมด      : {len(df)}")
print(f"เฉลี่ยรูปต่อ group    : {len(df) / df['group_key'].nunique():.2f}")


In [ ]:
# ตัวอย่างแถวที่ไม่เข้า pattern ไหนเลย -> ควรเป็น GAN/diffusion/FFHQ ที่ generate อิสระ
unmatched = df[df["group_pattern"] == "unmatched_unique"]
print(f"จำนวนแถวที่เป็น unique group (ไม่เข้า pattern): {len(unmatched)}")
print("\nตัวอย่าง path ของกลุ่มนี้ (สุ่มดู 15 แถว) -> เช็คว่าถูกต้องไหมว่าไม่ควรมี identity ซ้ำ:")
unmatched[["Image Path"]].sample(min(15, len(unmatched)), random_state=RANDOM_STATE)


### 4.1 Breakdown ของ unmatched ตาม top-level dataset folder

ข้อมูลเต็มมีขนาดใหญ่ (หลักล้านแถว) การสุ่มดู 15 แถวอาจไม่พอจับทุก pattern
เซลล์นี้แจกแจง `unmatched_unique` ตาม category ของ path เพื่อดูภาพรวมว่าเหลือ folder ไหนบ้าง
และในแต่ละ folder ยังเป็น "synthetic อิสระ" จริงไหม (ไม่ควรมี id/ชื่อไฟล์ซ้ำเป็นชุดๆ)

In [ ]:
def top_level_category(path: str) -> str:
    # ตัด root "/AI_Face_imagesV2/" ออก แล้วเอา 2 ระดับแรกของ path มาเป็น category
    parts = path.strip("/").split("/")
    if parts and parts[0] == "AI_Face_imagesV2":
        parts = parts[1:]
    return "/".join(parts[:2]) if len(parts) >= 2 else "/".join(parts)

unmatched = unmatched.copy()
unmatched["category"] = unmatched["Image Path"].apply(top_level_category)

cat_counts = unmatched["category"].value_counts()
print("จำนวนแถว unmatched แยกตาม category:")
print(cat_counts)

print("\nตัวอย่าง 3 path ต่อ category (เช็คว่าแต่ละ category ควรเป็น unique จริงไหม):")
for cat in cat_counts.index:
    sample_paths = unmatched.loc[unmatched["category"] == cat, "Image Path"].head(3).tolist()
    print(f"\n[{cat}]  (n={cat_counts[cat]})")
    for p in sample_paths:
        print(f"   {p}")


In [ ]:
# ตัวอย่างกลุ่มที่มีหลายรูป -> เช็คว่ารูปในกลุ่มเดียวกันมาจาก identity/วิดีโอเดียวกันจริง
example_group = group_sizes[group_sizes > 1].index[0]
df[df["group_key"] == example_group][["Image Path", "group_key", "group_pattern", "Target"]]


## 5. (Optional) เช็คว่าไฟล์รูปมีอยู่จริง

ข้ามอัตโนมัติถ้า `IMAGE_ROOT` เป็น `None`

In [ ]:
if IMAGE_ROOT is not None:
    def file_exists(rel_path: str) -> bool:
        p = IMAGE_ROOT / rel_path.lstrip("/")
        return p.exists()

    df["has_image"] = df["Image Path"].apply(file_exists)
    missing = df[~df["has_image"]]
    print(f"Missing image files: {len(missing)} / {len(df)}")
    if len(missing) > 0:
        print(missing["Image Path"].head(20))
else:
    print("ข้ามการเช็คไฟล์รูป (IMAGE_ROOT = None)")


## 6. Stratified Group K-Fold

- **group** = `group_key` (identity/วิดีโอ) → กันรูปจากคนหรือวิดีโอเดียวกันไม่ให้แยกไปคนละ fold
- **stratify** = `Target` (real=0 / fake=1) → คุมสัดส่วน real/fake ให้ใกล้เคียงกันทุก fold
- 10-fold ตามที่ยืนยัน

In [ ]:
X = df.index.values
y = df["Target"].values
groups = df["group_key"].values

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

df["fold"] = -1
for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
    df.loc[val_idx, "fold"] = fold_idx

print("จำนวน record ต่อ fold:")
print(df["fold"].value_counts().sort_index())


## 7. ตรวจสอบว่าไม่มี identity/video leakage ระหว่าง fold

In [ ]:
leak_found = False
for fold in range(N_SPLITS):
    val_groups = set(df.loc[df["fold"] == fold, "group_key"])
    train_groups = set(df.loc[df["fold"] != fold, "group_key"])
    overlap = val_groups & train_groups
    if overlap:
        leak_found = True
        print(f"Fold {fold}: ⚠️ LEAKAGE พบ group ซ้ำ {len(overlap)} กลุ่ม")
    else:
        print(f"Fold {fold}: OK — validation {len(val_groups)} groups, ไม่มีการซ้ำกับ train")

print("\n✅ ไม่พบ leakage ในทุก fold" if not leak_found else "\n❌ พบ leakage โปรดตรวจสอบ")


## 8. ตรวจสอบสัดส่วน Target และ Intersection ในแต่ละ fold

In [ ]:
import matplotlib.pyplot as plt

target_dist = df.groupby(["fold", "Target"]).size().unstack(fill_value=0)
target_dist["total"] = target_dist.sum(axis=1)
target_dist["fake_ratio"] = (target_dist[1] / target_dist["total"]).round(3) if 1 in target_dist.columns else np.nan
print("การกระจายของ Target ต่อ fold:")
display(target_dist)

# ไม่ได้ stratify ตาม Intersection แต่ดูไว้เพื่อรู้ทัน ว่า fairness subgroup เอียงไปทาง fold ไหนหรือไม่
intersection_dist = df.groupby(["fold", "Intersection"]).size().unstack(fill_value=0)
print("\nการกระจายของ Intersection (gender x skin tone group) ต่อ fold — ไว้ดูเฉยๆ ไม่ได้ควบคุมโดยตรง:")
display(intersection_dist)


### 8.1 กราฟเปรียบเทียบสัดส่วน Target ระหว่าง fold

ใช้ **สัดส่วน (proportion)** แทนจำนวนดิบ เพราะแต่ละ fold มีขนาดใกล้เคียงกันอยู่แล้ว
การดูสัดส่วนจะเห็นชัดกว่าว่า fold ไหน "เอียง" ผิดปกติหรือไม่

In [ ]:
target_prop = target_dist[[c for c in target_dist.columns if c not in ("total", "fake_ratio")]]
target_prop = target_prop.div(target_prop.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
target_prop.plot(kind="bar", stacked=False, ax=ax, width=0.75)
ax.set_xlabel("Fold")
ax.set_ylabel("Proportion")
ax.set_title("Target distribution per fold (proportion)")
ax.legend(title="Target", labels=[f"Target={c}" for c in target_prop.columns])
ax.set_xticklabels(target_prop.index, rotation=0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


### 8.2 กราฟเปรียบเทียบสัดส่วน Intersection ระหว่าง fold

กราฟแท่งกลุ่ม (grouped bar) — แต่ละกลุ่มสีคือ fold, แต่ละแท่งย่อยคือค่า Intersection
ถ้าทุก fold มีความสูงของแท่งสีเดียวกันใกล้เคียงกัน แปลว่าการกระจายเหมือนกันทุก fold

In [ ]:
intersection_prop = intersection_dist.div(intersection_dist.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
intersection_prop.T.plot(kind="bar", ax=ax, width=0.8)
ax.set_xlabel("Intersection")
ax.set_ylabel("Proportion")
ax.set_title("Intersection distribution per fold (proportion)")
ax.legend(title="Fold", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_xticklabels(intersection_prop.columns, rotation=0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


### 8.3 Heatmap สัดส่วน Intersection x Fold

มองภาพรวมทีเดียวว่ามี cell ไหนสีเข้ม/อ่อนผิดปกติจาก pattern โดยรวมหรือไม่

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(intersection_prop.values, aspect="auto", cmap="viridis")

ax.set_xticks(range(len(intersection_prop.columns)))
ax.set_xticklabels(intersection_prop.columns)
ax.set_yticks(range(len(intersection_prop.index)))
ax.set_yticklabels(intersection_prop.index)
ax.set_xlabel("Intersection")
ax.set_ylabel("Fold")
ax.set_title("Intersection proportion heatmap (fold x intersection)")

for i in range(intersection_prop.shape[0]):
    for j in range(intersection_prop.shape[1]):
        ax.text(j, i, f"{intersection_prop.values[i, j]:.3f}",
                ha="center", va="center", color="white", fontsize=8)

fig.colorbar(im, ax=ax, label="Proportion")
plt.tight_layout()
plt.show()


## 9. ทดสอบสมมติฐานทางสถิติ (Chi-square test)

**สมมติฐาน:**
- H0: สัดส่วนของตัวแปร (Target / Intersection) **เหมือนกัน** ในทุก fold (independent จาก fold)
- H1: สัดส่วนแตกต่างกันอย่างมีนัยสำคัญระหว่าง fold

ใช้ **Chi-square test of independence** กับตาราง contingency (fold x ตัวแปร) ที่ alpha = 0.05

**ข้อควรระวัง:** ข้อมูลมีขนาดใหญ่มาก (>1.3 ล้านแถว) ซึ่ง chi-square test ไวต่อขนาดตัวอย่างมาก
แม้ต่างกันเพียงเล็กน้อยในทางปฏิบัติ (practically negligible) ก็อาจได้ p-value ต่ำกว่า 0.05 ได้ง่าย
จึงต้องดู **effect size (Cramér's V)** ควบคู่ไปด้วยเพื่อตัดสินว่าความต่างนั้น "มีนัยสำคัญทางสถิติ" จริง
หรือแค่ "detectable เพราะ n เยอะ" — โดยทั่วไป Cramér's V < 0.1 ถือว่า negligible/very weak association

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(chi2_stat, contingency_table):
    n = contingency_table.values.sum()
    r, k = contingency_table.shape
    return np.sqrt(chi2_stat / (n * (min(r, k) - 1)))


def run_chi2_test(contingency_table: pd.DataFrame, label: str, alpha: float = 0.05):
    chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)
    v = cramers_v(chi2_stat, contingency_table)

    print(f"=== {label} ===")
    print(f"chi2 statistic : {chi2_stat:.4f}")
    print(f"degrees of freedom : {dof}")
    print(f"p-value        : {p_value:.6g}")
    print(f"Cramer's V     : {v:.5f}")

    if p_value < alpha:
        print(f"\n-> p-value < {alpha}: ปฏิเสธ H0 (สัดส่วนแตกต่างกันอย่างมีนัยสำคัญทางสถิติ)")
    else:
        print(f"\n-> p-value >= {alpha}: ไม่มีหลักฐานเพียงพอที่จะปฏิเสธ H0 (สัดส่วนไม่ต่างกันอย่างมีนัยสำคัญ)")

    if v < 0.1:
        print("-> Cramer's V < 0.1: ความสัมพันธ์ระหว่าง fold กับตัวแปรนี้ negligible/very weak")
        print("   (ต่อให้ p-value ต่ำ ก็แปลว่าความต่างเล็กน้อยมากในทางปฏิบัติ ไม่ได้กระทบการใช้งานจริง)")
    elif v < 0.3:
        print("-> Cramer's V อยู่ในระดับ weak-moderate ควรพิจารณาตรวจสอบเพิ่มเติม")
    else:
        print("-> Cramer's V สูง บ่งชี้ว่า fold มีความไม่สมดุลจริงในทางปฏิบัติ ควรทบทวนวิธีแบ่ง fold")

    print()
    return chi2_stat, p_value, dof, v


ALPHA = 0.05

target_counts = target_dist[[c for c in target_dist.columns if c not in ("total", "fake_ratio")]]
run_chi2_test(target_counts, "Target x Fold", alpha=ALPHA)

run_chi2_test(intersection_dist, "Intersection x Fold", alpha=ALPHA)


## 10. บันทึกผลลัพธ์

In [ ]:
OUTPUT_CSV = CSV_PATH.with_name(CSV_PATH.stem + "_with_folds.csv")
df.to_csv(OUTPUT_CSV, index=False)
print(f"บันทึกไฟล์แล้ว: {OUTPUT_CSV}")
df.head()


---
### หมายเหตุ

- ถ้าพบว่า `unmatched_unique` มีจำนวนมากผิดปกติ หรือเจอ path pattern ใหม่ที่ควรมี identity ซ้ำแต่ไม่ถูกจับกลุ่ม
  ให้เพิ่ม regex ใหม่ใน `PATTERNS` (เซลล์ที่ 3) แล้วรันใหม่ตั้งแต่เซลล์นั้น
- ตรวจสอบแล้วว่า GANs/ProGAN, StyleGAN, StyleGAN2 (SFHQ), stylegan3, taming_transformer_VQGAN, latent_diffusion,
  Real/FFHQ, AttGAN, Palette, STARGAN, STGAN, MSG_STYLE_GAN, MMD_GAN เป็นภาพ synthetic ที่ generate อิสระต่อกันจริง
  (ไม่มี id ซ้ำ) จึงปล่อยให้เป็น unique group ต่อภาพได้อย่างปลอดภัย — ทุก category อื่น (DCFACe, celebdf, ff++, dfdc,
  dfd, imdb_wiki, StableDiffusion1.5/Inpainting, CommercialTools) ผ่านการตรวจสอบและแก้ regex จนครอบคลุมครบแล้ว
- ถ้าอยาก stratify ด้วย `Intersection` ร่วมด้วย (คุม fairness ในแต่ละ fold) สามารถเปลี่ยน `y` ในขั้นตอนที่ 6 เป็น
  `df["Target"].astype(str) + "_" + df["Intersection"].astype(str)` ได้ แต่ต้องระวัง — บาง subgroup อาจมีจำนวนน้อยจนแบ่ง 10 fold ไม่ลงตัว
- ตอน train ให้ filter ด้วยคอลัมน์ `fold` ได้เลย เช่น
  `train_df = df[df["fold"] != k]`, `val_df = df[df["fold"] == k]`


---
# Part B: วิเคราะห์ train/test split ที่มีอยู่แล้ว (`train_data.csv`, `test_data.csv`)

ส่วนนี้แยกจาก Part A (การสร้าง fold เอง) — ใช้วิเคราะห์ไฟล์ train/test ที่ **มีคนแบ่งไว้ให้แล้ว**
เพื่อดูว่า:
1. แบ่งสัดส่วน train/test เท่าไหร่
2. มี **identity/video leakage** ระหว่าง train กับ test หรือไม่ (ใช้ `extract_group()` function เดียวกับ Part A)
3. การกระจายของ `Target`, `Intersection`, `Skin Tone Group`, `Predicted Gender`, `Predicted Age` ใน train vs test ต่างกันแค่ไหน

**หมายเหตุ:** เซลล์ในส่วนนี้เรียกใช้ `extract_group()`, `run_chi2_test()`, `cramers_v()` ที่นิยามไว้ใน Part A ด้านบน
ต้องรัน Part A (อย่างน้อยเซลล์ที่นิยามฟังก์ชันเหล่านี้) ก่อน จึงจะรันส่วนนี้ได้

## 11. โหลดข้อมูล train/test

In [ ]:
TRAIN_CSV = Path("/media/hdd2/code/MS-Workspace/train_data.csv")
TEST_CSV = Path("/media/hdd2/code/MS-Workspace/test_data.csv")

df_train = pd.read_csv(TRAIN_CSV)
df_test = pd.read_csv(TEST_CSV)

df_train.columns = [c.strip() for c in df_train.columns]
df_test.columns = [c.strip() for c in df_test.columns]

df_train["split"] = "train"
df_test["split"] = "test"

n_train, n_test = len(df_train), len(df_test)
n_total = n_train + n_test

print(f"Train rows : {n_train:,}")
print(f"Test rows  : {n_test:,}")
print(f"Total rows : {n_total:,}")
print(f"Test ratio : {n_test / n_total:.2%}")

print("\nTrain columns:", list(df_train.columns))
print("Test columns :", list(df_test.columns))

df_split = pd.concat([df_train, df_test], ignore_index=True, sort=False)
df_split.head()


## 12. ตรวจสอบว่ารูปเดียวกัน (path ซ้ำ) หลุดไปอยู่ทั้ง train และ test ไหม

กรณีนี้ตรงไปตรงมาที่สุด: path เดียวกันเป๊ะ ไม่ควรปรากฏทั้งสองฝั่ง

In [ ]:
dup_paths = df_split[df_split.duplicated("Image Path", keep=False)]
print(f"จำนวนแถวที่ Image Path ซ้ำกัน (รวมทุกครั้งที่ปรากฏ): {len(dup_paths)}")

if len(dup_paths) > 0:
    path_split_count = dup_paths.groupby("Image Path")["split"].nunique()
    n_cross_split = (path_split_count > 1).sum()
    print(f"จำนวน path ที่ปรากฏทั้งใน train และ test: {n_cross_split}")
    if n_cross_split > 0:
        print("\n⚠️ พบรูปเดียวกันอยู่ทั้ง train และ test — ควรตัดออกจากฝั่งใดฝั่งหนึ่งก่อนใช้งานจริง")
        display(dup_paths[dup_paths["Image Path"].isin(path_split_count[path_split_count > 1].index)]
                .sort_values("Image Path")[["Image Path", "split", "Target"]].head(20))
else:
    print("✅ ไม่มี path ซ้ำกันเลยระหว่าง/ภายใน train-test")


## 13. ตรวจสอบ identity / video-level leakage ระหว่าง train และ test

ใช้ `extract_group()` function เดียวกับที่ใช้สร้าง fold ใน Part A (ครอบคลุม DCFACe, celebdf, ff++, dfdc, dfd,
imdb_wiki/StableDiffusion, CommercialTools) เพื่อเช็คว่าคนเดียวกัน/วิดีโอเดียวกัน (path ไม่เหมือนกันเป๊ะ
แต่มาจาก identity เดียวกัน) หลุดไปอยู่ทั้ง train และ test หรือไม่ — ถ้าพบ ถือเป็น leakage ที่ร้ายแรงกว่า
path ซ้ำธรรมดา เพราะโมเดลอาจ "จำ" identity นั้นได้จากตอนเทรน แล้วไปทำคะแนนสูงผิดปกติตอน evaluate บน test

In [ ]:
groups_and_labels_split = [
    extract_group(path, idx) for idx, path in enumerate(df_split["Image Path"])
]
df_split["group_key"] = [g for g, _ in groups_and_labels_split]
df_split["group_pattern"] = [lbl for _, lbl in groups_and_labels_split]

group_split_map = df_split.groupby("group_key")["split"].nunique()
leaking_groups = group_split_map[group_split_map > 1]

print(f"จำนวน identity/video group ทั้งหมด        : {df_split['group_key'].nunique():,}")
print(f"จำนวน group ที่ปรากฏทั้งใน train และ test  : {len(leaking_groups):,}")

if len(leaking_groups) > 0:
    n_rows_affected = df_split[df_split["group_key"].isin(leaking_groups.index)].shape[0]
    print(f"จำนวนแถวที่ได้รับผลกระทบ (อยู่ใน group ที่ leak) : {n_rows_affected:,}")
    print("\n⚠️ พบ identity/video leakage ระหว่าง train/test")
    print("ตัวอย่าง group ที่ leak (เรียงตาม group_key):")
    display(df_split[df_split["group_key"].isin(leaking_groups.index)]
            .sort_values("group_key")[["Image Path", "group_key", "group_pattern", "split", "Target"]].head(30))
else:
    print("\n✅ ไม่พบ identity/video leakage ระหว่าง train และ test")


### 13.1 Leakage กระจุกอยู่ที่หมวดไหนบ้าง

แจกแจงแถวที่ leak ตาม `group_pattern` (หมวดข้อมูล) เทียบกับจำนวนทั้งหมดของหมวดนั้น
เพื่อดูว่าหมวดไหนได้รับผลกระทบหนักสุด — โครงสร้างของ `extract_group()` การันตีว่าแถวที่เป็น
`unmatched_unique` (GAN/diffusion synthetic ที่ไม่มี identity) จะไม่มีทาง leak ได้เลย เพราะแต่ละแถว
กลายเป็น group เดี่ยวของตัวเอง (unique ต่อแถว) ตั้งแต่ต้น

In [ ]:
if len(leaking_groups) > 0:
    leaking_rows = df_split[df_split["group_key"].isin(leaking_groups.index)]

    print("จำนวนแถวที่ leak แยกตาม group_pattern (หมวดข้อมูล):")
    leak_counts_by_pattern = leaking_rows["group_pattern"].value_counts()
    print(leak_counts_by_pattern)

    total_by_pattern = df_split["group_pattern"].value_counts()
    leak_ratio = (leak_counts_by_pattern / total_by_pattern.reindex(leak_counts_by_pattern.index)).sort_values(ascending=False)
    print("\nสัดส่วนแถวที่ leak เทียบกับจำนวนทั้งหมดของแต่ละหมวด (ยิ่งใกล้ 1.0 ยิ่งแย่ = แทบทั้งหมวดถูกแบ่งมั่ว):")
    print(leak_ratio.round(3))
else:
    print("ไม่มี leakage ให้แจกแจง")


### 13.2 ทางแก้เบื้องต้น: สร้าง test set ที่ไม่มี identity ซ้ำกับ train

ตัด test row ใดๆ ที่ `group_key` ไปซ้ำกับที่มีอยู่ใน train ออก จะได้ test set ที่ **ไม่มี leakage แน่นอน**
แล้วดูว่าเหลือข้อมูลพอใช้ evaluate ได้จริงแค่ไหน — ถ้าเหลือน้อยเกินไป (เช่น < 50% ของ test เดิม)
แนะนำให้พิจารณาทิ้ง split นี้แล้วใช้วิธี group-based split เอง (แบบ Part A ด้านบน) แทนไปเลย
จะควบคุมได้ทั้งสัดส่วนและป้องกัน leakage ตั้งแต่ต้น

In [ ]:
groups_in_train = set(df_split.loc[df_split["split"] == "train", "group_key"])

test_mask_clean = (df_split["split"] == "test") & (~df_split["group_key"].isin(groups_in_train))
df_test_clean = df_split[test_mask_clean].copy()

n_test_original = (df_split["split"] == "test").sum()
n_test_clean = len(df_test_clean)

print(f"Test เดิม                          : {n_test_original:,} แถว")
print(f"Test หลังตัด identity ที่ซ้ำกับ train : {n_test_clean:,} แถว ({n_test_clean / n_test_original:.1%} ของเดิม)")

if n_test_clean / n_test_original < 0.5:
    print("\n⚠️ เหลือข้อมูลน้อยกว่าครึ่งของ test เดิม — แนะนำพิจารณาใช้ group-based split เองแทน (Part A)")
    print("   แทนที่จะพยายามซ่อม split นี้ เพราะข้อมูลที่เหลือใช้ evaluate อาจไม่ครอบคลุมพอ")
else:
    print("\nเหลือข้อมูลมากกว่าครึ่งหนึ่ง — พอใช้เป็น test set ชั่วคราวที่ไม่มี leakage ได้")

print("\nการกระจายของ Target ใน test เดิม vs test ที่ clean แล้ว:")
display(pd.DataFrame({
    "test (เดิม)": df_split.loc[df_split["split"] == "test", "Target"].value_counts(normalize=True).round(3),
    "test (clean)": df_test_clean["Target"].value_counts(normalize=True).round(3),
}))


## 14. เปรียบเทียบการกระจายของข้อมูลระหว่าง train/test

### 14.1 Target

In [ ]:
target_dist_split = df_split.groupby(["split", "Target"]).size().unstack(fill_value=0)
target_dist_split["total"] = target_dist_split.sum(axis=1)
print("การกระจายของ Target:")
display(target_dist_split)

target_counts_split = target_dist_split[[c for c in target_dist_split.columns if c != "total"]]
target_prop_split = target_counts_split.div(target_counts_split.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(6, 5))
target_prop_split.plot(kind="bar", ax=ax, width=0.6)
ax.set_xlabel("Split")
ax.set_ylabel("Proportion")
ax.set_title("Target distribution: train vs test")
ax.legend(title="Target", labels=[f"Target={c}" for c in target_prop_split.columns])
ax.set_xticklabels(target_prop_split.index, rotation=0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


### 14.2 Intersection

In [ ]:
intersection_dist_split = df_split.groupby(["split", "Intersection"]).size().unstack(fill_value=0)
print("การกระจายของ Intersection:")
display(intersection_dist_split)

intersection_prop_split = intersection_dist_split.div(intersection_dist_split.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
intersection_prop_split.T.plot(kind="bar", ax=ax, width=0.7)
ax.set_xlabel("Intersection")
ax.set_ylabel("Proportion")
ax.set_title("Intersection distribution: train vs test")
ax.legend(title="Split")
ax.set_xticklabels(intersection_prop_split.columns, rotation=0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


### 14.3 Skin Tone Group และ Predicted Gender

In [ ]:
skintone_dist_split = df_split.groupby(["split", "Skin Tone Group"]).size().unstack(fill_value=0)
gender_dist_split = df_split.groupby(["split", "Predicted Gender"]).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

skintone_prop_split = skintone_dist_split.div(skintone_dist_split.sum(axis=1), axis=0)
skintone_prop_split.T.plot(kind="bar", ax=axes[0], width=0.7)
axes[0].set_title("Skin Tone Group: train vs test")
axes[0].set_xlabel("Skin Tone Group")
axes[0].set_ylabel("Proportion")
axes[0].legend(title="Split")
axes[0].set_xticklabels(skintone_prop_split.columns, rotation=0)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

gender_prop_split = gender_dist_split.div(gender_dist_split.sum(axis=1), axis=0)
gender_prop_split.T.plot(kind="bar", ax=axes[1], width=0.6)
axes[1].set_title("Predicted Gender: train vs test")
axes[1].set_xlabel("Predicted Gender")
axes[1].set_ylabel("Proportion")
axes[1].legend(title="Split")
axes[1].set_xticklabels(gender_prop_split.columns, rotation=0)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


### 14.4 Predicted Age

`Predicted Age` เป็นตัวแปร **categorical** (bin ช่วงอายุ รหัส 0-4) ไม่ใช่ตัวเลขอายุต่อเนื่อง
จึงดูแบบ bar chart สัดส่วนเหมือน Gender/Skin Tone Group ไม่ใช้ boxplot

In [ ]:
age_dist_split = df_split.groupby(["split", "Predicted Age"]).size().unstack(fill_value=0)
print("การกระจายของ Predicted Age:")
display(age_dist_split)

age_prop_split = age_dist_split.div(age_dist_split.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(8, 5))
age_prop_split.T.plot(kind="bar", ax=ax, width=0.7)
ax.set_xlabel("Predicted Age (bin)")
ax.set_ylabel("Proportion")
ax.set_title("Predicted Age distribution: train vs test")
ax.legend(title="Split")
ax.set_xticklabels(age_prop_split.columns, rotation=0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## 15. ทดสอบสมมติฐานทางสถิติ: train vs test (alpha = 0.05)

เหมือน Part A แต่เทียบแค่ 2 กลุ่ม (train / test) แทนที่จะเป็นหลาย fold — ตัวแปรทั้งหมดใน Part B เป็น
categorical (`Target`, `Intersection`, `Skin Tone Group`, `Predicted Gender`, `Predicted Age` เป็น bin)
จึงใช้ **Chi-square test** กับทุกตัวแปร

In [ ]:
run_chi2_test(target_counts_split, "Target x Split", alpha=ALPHA)
run_chi2_test(intersection_dist_split, "Intersection x Split", alpha=ALPHA)
run_chi2_test(skintone_dist_split, "Skin Tone Group x Split", alpha=ALPHA)
run_chi2_test(gender_dist_split, "Predicted Gender x Split", alpha=ALPHA)
run_chi2_test(age_dist_split, "Predicted Age x Split", alpha=ALPHA)


---
### สรุป Part B

- ถ้า section 12/13 เจอ leakage (path ซ้ำ หรือ identity/video ซ้ำข้าม train/test) นี่คือปัญหาที่ควรแก้ก่อนใช้ split นี้
  ไปเทรน/ประเมินโมเดลจริง เพราะ metric บน test set จะดูดีเกินจริง (overly optimistic)
- ถ้า chi-square/Mann-Whitney ใน section 15 ให้ p-value ต่ำ ให้ดู effect size (Cramér's V) ควบคู่เสมอ — ข้อมูลขนาดใหญ่
  (แสนถึงล้านแถว) ทำให้ p-value ต่ำได้ง่ายแม้ความต่างจะเล็กน้อยในทางปฏิบัติ (หลักการเดียวกับ Part A)
